# JWST SBF Setup
Version 24 July 2026, J. Jensen

- The i2d files have the following extensions: [1] SCI, [2] ERR, [3] VAR_POISSON, [4] VAR_RNOISE, [5] VAR_FLAT, [6] WHT, [7] CON <BR>

This version 
- sets up the metadata file (json)
- extracts sub-images centered on the target galaxies (and associated dmask regions)
- sets up old-fashioned files for traditional SBF analysis (flucin, centers, calibrate.dat, etc.)
- it estimates the FWHM and saturation level


In [1]:
### Install the required python packages
import sys, os
import datetime

pysbf_path = "/Users/joejensen/data/sbf/"
config_path = "/Users/joejensen/data/sbf/pysbf/config/"
sys.path.insert(0, pysbf_path)
from pysbf import *
from astropy.time import Time

import astropy.io.fits as fits
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pylab as py
import json

configFolder = pysbf_path + "pysbf/config/sextractor/"
version = "SBFsetup-JWST: 2026-07-24"

now = datetime.now()

In [2]:
# input values
extfrac = 0.1    # uncertainty in extinction
pixscale_sw = 0.031 # arcsec/pixel
pixscale_lw = 0.063 # arcsec/pixel
gain = 1.0       # electrons/DN
H0 = 73.         # Hubble constant for distance estimates
imagesize = 2048. # extracted image size
fwhm = {"F090W":1.5,"F150W":1.7,"F277W":1.7,"F356W":1.9}
seeing = {"F090W":0.047,"F150W":0.053,"F277W":0.107,"F356W":0.116}
pixscale = {"F090W":0.031,"F150W":0.031,"F277W":0.063,"F356W":0.063}



## Object Initialization

In [3]:
# use this if flicker-corrected images are in a separate _flicker directory
flicker = False

# Use this if the NED info cache file needs to be reset
# resetCache = False

# Use this to overwrite the output files (flucin, calibrate.dat, centers.dat)
overwrite = False

In [4]:
ProgramID = '3055'
overwrite = True

In [5]:
# Set these for the specific data set and galaxy. 
# Note that for JWST the "basename" is the combined NIRCam image, and the name is the individual galaxy within 
# that image. For now the center has to be set manually so we know which galaxy to use.
# The centers in the LW and SW channels are set independently.

# Coma SBF Survey GO-5989 (use flicker correction)
if ProgramID == '5989':
    JWST_root = '/Users/joejensen/data/jwst-5989/'
    filters = ['F150W','F356W']
    r1=str(400)
    flicker = True
    #name = 'pgc44539'; basename='n4860'; xcenter_sw=0; ycenter_sw=0; xcenter_lw=1293; ycenter_lw=860; r1=str(300) # ngc 4860
    # name = 'pgc44578'; basename='n4865'; xcenter_lw=1690; ycenter_lw=1417 # ngc 4865
    ###name = 'pgc44574'; basename='n4865'; xcenter_lw=1412; ycenter_lw=1922
    # name = 'pgc44560'; basename='n4865'; xcenter_lw=405; ycenter_lw=1062 
    # name = 'pgc44587'; basename='n4869'; xcenter_lw=3677; ycenter_lw=1336; flicker=True # ngc 4869

# TRGB Calibrators (GO-3055)
elif ProgramID == '3055':
    JWST_root = '/Users/joejensen/data/jwst-3055/'
    filters = ['F090W','F150W','F277W','F356W']
    r1=str(600)
    flicker = False
    name = 'n1380'; basename='n1380'; xcenter_sw=6614; ycenter_sw=1166; xcenter_lw=3259; ycenter_lw=537 
    # name = 'n1399'; basename='n1399'; xcenter_sw=6543; ycenter_sw=3208; xcenter_lw=3234; ycenter_lw=1551
    #name = 'n1404'; basename='n1404'; xcenter_sw=8784; ycenter_sw=3217; xcenter_lw=4345; ycenter_lw=1545 
    #name = 'n1549'; basename='n1549'; xcenter_sw=8458; ycenter_sw=3116; xcenter_lw=4183; ycenter_lw=1496
    #name = 'n3379'; basename='n3379'; xcenter_sw=8480; ycenter_sw=3185; xcenter_lw=4195; ycenter_lw=1530
    #name = 'n4374'; basename='n4374'; xcenter_sw=8476; ycenter_sw=3186; xcenter_lw=4192; ycenter_lw=1532 
    #name = 'n4406'; basename='n4406'; xcenter_sw=8486; ycenter_sw=3147; xcenter_lw=4198; ycenter_lw=1512 
    #name = 'n4472'; basename='n4472'; xcenter_sw=8809; ycenter_sw=3323; xcenter_lw=4359; ycenter_lw=1598 
    #name = 'n4486'; basename='n4486'; xcenter_sw=1443; ycenter_sw=1210; xcenter_lw=693; ycenter_lw=581 
    #name = 'n4552'; basename='n4552'; xcenter_sw=8477; ycenter_sw=3186; xcenter_lw=4193; ycenter_lw=1531 
    #name = 'n4621'; basename='n4621'; xcenter_sw=8476; ycenter_sw=3185; xcenter_lw=4193; ycenter_lw=1531 
    #name = 'n4636'; basename='n4636'; xcenter_sw=8460; ycenter_sw=3172; xcenter_lw=4185; ycenter_lw=1526 
    #name = 'n4649'; basename='n4649'; xcenter_sw=8493; ycenter_sw=1266; xcenter_lw=4193; ycenter_lw=579 
    #name = 'n4697'; basename='n4697'; xcenter_sw=8461; ycenter_sw=3173; xcenter_lw=4185; ycenter_lw=1525 

elif ProgramID == '7034':
    JWST_root = '/Users/joejensen/data/jwst-7034/'
    filters = ['F090W','F150W','F277W','F356W']
    r1=str(600)
    flicker = False
    #name = 'n0524'; basename='n0524'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n1201'; basename='n1201'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n1344'; basename='n1344'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n1374'; basename='n1374'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'ic2006'; basename='ic2006'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n3643'; basename='n3643'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n3941'; basename='n3941'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n4036'; basename='n4036'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n4125'; basename='n4125'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n4386'; basename='n4386'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 
    #name = 'n'; basename='n'; xcenter_sw=; ycenter_sw=; xcenter_lw=; ycenter_lw= 

else:
    print('Program ID ',ProgramID,' not found.')

In [6]:
# Name convention for filter labels with legacy software:
filterlabel = {'F090W': 'c', 'F150W': 'd', 'F277W': 'e', 'F356W': 'f'}

# Name conventions for files:
if flicker:
    suffix = f'{basename}_flicker/'
    tag = '_flicker_i2d.fits'
else:
    suffix = f'{basename}/'
    tag = '_i2d.fits'

fits_files = {f: JWST_root + suffix + basename + f'_{f}' + tag for f in filters}

dmask_sw = JWST_root + suffix + basename + '_sw.dmask'
dmask_lw = JWST_root + suffix + basename + '_lw.dmask'
print(dmask_sw, dmask_lw)

for filter in filters:
    filename = fits_files[filter]
    if not os.path.exists(filename):
        print(f'File {filename} does not exist.')
    else:
        print(f'Using file {filename}')

SBF_root = JWST_root+'SBF'
if not os.path.isdir(SBF_root):
    os.makedirs(SBF_root)
if not os.path.isdir(SBF_root+'/'+name):
    os.makedirs(SBF_root+'/'+name)

datafile = SBF_root+'/'+name+'/'+name+'_data.json'

#fiterp = {filter: SBF_root+'/'+name+'/'+name+'_'+filter+'.fiterpolate' for filter in filterlabel}
resid = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.resid' for f in filterlabel}
fiterp = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.fiterpolate' for f in filterlabel}
model = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.prf' for f in filterlabel}
flucin = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.flucin' for f in filterlabel}
dpar = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.dpar' for f in filterlabel}
inpar = {f: SBF_root+'/'+name+'/'+name+filterlabel[f]+'.inpar' for f in filterlabel}

calibrate = SBF_root+'/'+name+'/calibrate.dat'
centers = SBF_root+'/'+name+'/centers.dat'


/Users/joejensen/data/jwst-3055/n1380/n1380_sw.dmask /Users/joejensen/data/jwst-3055/n1380/n1380_lw.dmask
Using file /Users/joejensen/data/jwst-3055/n1380/n1380_F090W_i2d.fits
Using file /Users/joejensen/data/jwst-3055/n1380/n1380_F150W_i2d.fits
Using file /Users/joejensen/data/jwst-3055/n1380/n1380_F277W_i2d.fits
Using file /Users/joejensen/data/jwst-3055/n1380/n1380_F356W_i2d.fits


### Create cutout images

In [7]:
maxvalue = {}

for filter in filters:
    filename = fits_files[filter]

    if filter in ['F090W','F150W']:
        imagesize = 2048
        dmask = dmask_sw
        xcenter = xcenter_sw; ycenter = ycenter_sw
        new_xcen_sw = imagesize/2; new_ycen_sw = imagesize/2
    elif filter in ['F277W','F356W']:
        imagesize = 1024
        dmask = dmask_lw
        xcenter = xcenter_lw; ycenter = ycenter_lw
        new_xcen_lw = imagesize/2; new_ycen_lw = imagesize/2
    else:
        print('Filter not supported.')
    
    outputfile = SBF_root+'/'+name+'/'+name+filterlabel[filter]+'.fits'
    outputmask = SBF_root+'/'+name+'/'+name+filterlabel[filter]+'.dmask'
    #outputfile = SBF_root+'/'+name+'/'+name+'_'+filter+'.fits'
    #outputmask = SBF_root+'/'+name+'/'+name+'_'+filter+'.dmask'

    with fits.open(filename) as hdul:
        header = hdul[1].header 
        exptime = header['XPOSURE']
    
    monsta_script = """
        rd 1 '"""+filename+"""'
        rd 2 '"""+dmask+"""'
        clip 1 nan=0
        abx 1 all high=high silent
        typ high
        mc 1 """+str(exptime)+"""
        open 3 nx="""+str(imagesize)+""" ny="""+str(imagesize)+"""
        cop 4 3
        shift 1 dx="""+str(-xcenter)+""" dy="""+str(-ycenter)+"""
        shift 1 dx="""+str(imagesize/2)+""" dy="""+str(imagesize/2)+"""
        buf
        ai 3 1
        shift 2 dx="""+str(-xcenter)+""" dy="""+str(-ycenter)+"""
        shift 2 dx="""+str(imagesize/2)+""" dy="""+str(imagesize/2)+"""
        ai 4 2
        wd 3 """+outputfile+"""
        wd 4 """+outputmask+"""
    """
    if not os.path.exists(dmask):
        print(f"dmask {dmask} does not exist. Please make one first.")
    elif not os.path.exists(filename):
        print(f"input file {filename} does not exist. Please construct one or check the directory path.")
    else:
        run_monsta(monsta_script, 'monsta.pro', 'monsta.log')
        print(f"Extracted files {outputfile} and {outputmask}.")

        with open("monsta.log", "r") as f:
            lines = f.readlines()
            
        for l in lines:
            l0 = l.strip()
            l00 = l0.split()
            if l00:
                if "HIGH" in l0:
                    maxvalue[filter]=(float(l00[2]))
  

Extracted files /Users/joejensen/data/jwst-3055/SBF/n1380/n1380c.fits and /Users/joejensen/data/jwst-3055/SBF/n1380/n1380c.dmask.
Extracted files /Users/joejensen/data/jwst-3055/SBF/n1380/n1380d.fits and /Users/joejensen/data/jwst-3055/SBF/n1380/n1380d.dmask.
Extracted files /Users/joejensen/data/jwst-3055/SBF/n1380/n1380e.fits and /Users/joejensen/data/jwst-3055/SBF/n1380/n1380e.dmask.
Extracted files /Users/joejensen/data/jwst-3055/SBF/n1380/n1380f.fits and /Users/joejensen/data/jwst-3055/SBF/n1380/n1380f.dmask.


### Get Galaxy Info

In [8]:
Vext = 0

import requests
from astropy.io.votable import parse_single_table
from io import BytesIO
 
def get_ned_extinction(target):
    url = f"https://ned.ipac.caltech.edu/NED::API/ExtinctionAtTarget?TARGET={target.replace(' ', '+')}"
    response = requests.get(url)
    table = parse_single_table(BytesIO(response.content)).to_table()
    ext = {}
    for row in table:
        ext[row['ext_bandpass']] = float(row['a_lambda'])
    return ext

def get_ned_pgc(target):
    url = f"https://ned.ipac.caltech.edu/NED::API/CrossidsOfObject?TARGET={target.replace(' ', '+')}"
    response = requests.get(url)
    table = parse_single_table(BytesIO(response.content)).to_table()
    for row in table:
        obj_name = row['object_name']  # XML ID attribute, not name attribute
        if obj_name.startswith('PGC '):
            return int(obj_name.replace('PGC', '').strip())
    return None  # No PGC name found

def get_ned_coords(target):
    url = f"https://ned.ipac.caltech.edu/NED::API/OverviewOfObject?TARGET={target.replace(' ', '+')}"
    response = requests.get(url)
    table = parse_single_table(BytesIO(response.content)).to_table()
    for row in table:
        obj_lat = row['equ_j2000_lat']
        obj_lon = row['equ_j2000_lon']  # XML ID attribute, not name attribute
        return obj_lat, obj_lon
    return None # no coords found

def get_ned_velocity(target):
    url = f"https://ned.ipac.caltech.edu/NED::API/OverviewOfObject?TARGET={target.replace(' ', '+')}"
    response = requests.get(url)
    table = parse_single_table(BytesIO(response.content)).to_table()
    for row in table:
        v_sun = row['v_Sun']
        v_cmb = row['v_Sun_3K']  # XML ID attribute, not name attribute
        v_cmb_sig = row['unc_v_Sun_3K']
        return v_sun, v_cmb, v_cmb_sig
    return None # no coords found

def get_ned_type(target):
    url = f"https://ned.ipac.caltech.edu/NED::API/OverviewOfObject?TARGET={target.replace(' ', '+')}"
    response = requests.get(url)
    table = parse_single_table(BytesIO(response.content)).to_table()
    for row in table:
        morph = row['o_class_morph']
        return morph
    return None # no coords found

#    return None
    
pgc = get_ned_pgc(name)
print(f"PGC: {pgc}")

dec, ra = get_ned_coords(name)
print(f'RA, dec: {ra}, {dec}')

v_sun, v_cmb, v_cmb_sig = get_ned_velocity(name)
print(f'CMB Velocity: {int(v_cmb)} km/s')

morph = get_ned_type(name)
print(f'Morphological type: {morph}')

ext = get_ned_extinction(name)
#if name not in cache:
#    cache[name] = get_ned_extinction(name)
#    pickle.dump(cache, open(cache_file, 'wb'))
#ext = cache[name]
 
if ext is not None:
    Vext = ext['Landolt V']
    Bext = ext['Landolt B']
    psgext = ext['PS1 g']
    pszext = ext['PS1 z']
    dcgext = ext['DES g']
    dciext = ext['DES i']
    dczext = ext['DES z']

# JWST Values from https://svo2.cab.inta-csic.es/theory/fps/index.php?&mode=browse&gname=JWST&gname2=NIRCam2025
# Based on the extinction law from Fitzpatrick (1999) and improved by Indebetouw et al. (2005) in the infrared.
ext090 = 0.506 * Vext
ext150 = 0.229 * Vext
ext277 = 0.0897 * Vext
ext356 = 0.0669 * Vext
sigmaext150_356 = extfrac * (ext150-ext356)
sigmaext090 = extfrac * ext090
sigmaext150 = extfrac * ext150
sigmaext277 = extfrac * ext277
sigmaext356 = extfrac * ext356
print("Galactic extinction Landolt V: %.3f" % Vext)
print("Galactic extinction F150W: %.4f" % ext150)
print("Galactic extinction F356W: %.4f" % ext356)

distance = v_cmb/73
print(f"{name} distance ~ {distance:.0f} Mpc")

if distance < 100:
    renuc = 7
elif distance < 150:
    renuc = 5
elif distance < 180:
    renuc = 3
elif distance < 200:
    renuc = 2.5
elif distance < 230:
    renuc = 2
elif distance < 250:
    renuc = 1.5
else:
    renuc = 1.


PGC: 13318
RA, dec: 54.1148758, -34.9760337
CMB Velocity: 1781 km/s
Morphological type: SA0
Galactic extinction Landolt V: 0.056
Galactic extinction F150W: 0.0129
Galactic extinction F356W: 0.0038
n1380 distance ~ 24 Mpc


In [9]:
# Get some relevant information from the LEDA catalog (stored locally)
def loadLeda(leda_catalog):
    """Loading the HyperLeda catalog: 
    - http://leda.univ-lyon1.fr/
    - This catalog tabulates the proper information on local galaxies
    :return: the Leda catalog in the Pandas dataFrame format
    :rtype: Pandas ``dataFrame``
    """

    df = pd.read_csv(leda_catalog, delimiter=',')
    df = df.rename(columns=lambda x: x.strip())
                   
    return df.set_index('PGC')

v3k=0; v=0; vlg=0; ttype=None; gtype=None; bar=None; ring=None; companion=None; distance=0

if pgc is not None:
    Leda = loadLeda("LEDA.csv")
    if pgc in Leda.index:
        v3k = Leda.loc[pgc,"v3k"]
        v = Leda.loc[pgc,"v"]
        vlg = Leda.loc[pgc,"vlg"]
        ttype = Leda.loc[pgc,"t"]
        gtype = Leda.loc[pgc,"type"]
        bar = Leda.loc[pgc,"bar"]
        ring = Leda.loc[pgc,"ring"]
        companion = Leda.loc[pgc,"multiple"]
        distance = v3k / H0
    else:
        print('row not found')
        v3k=0; v=0; vlg=0; ttype=None; gtype=None; bar=None; ring=None; companion=None; distance=0

print(f'Heliocentric Velocity: {int(v)} km/s')
print(f'Local Group Velocity: {int(vlg)} km/s')
print(f'CMB Velocity: {int(v3k)} km/s')
print(f'T-type: {ttype}')
print(f'Bar: {bar}')
print(f'Ring: {ring}')
print(f'Companion: {companion}')


/var/folders/3d/j416_7vj7fn_vm9vp7jj0hmc0000gq/T/ipykernel_28561/2008086544.py:10: DtypeWarning: Columns (71,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(leda_catalog, delimiter=',')


Heliocentric Velocity: 1874 km/s
Local Group Velocity: 1787 km/s
CMB Velocity: 1770 km/s
T-type: -2.3
Bar: B
Ring: nan
Companion: nan


In [10]:
# The NED coordinates for these are wrong or between the main galaxy and a satellite    
#if name == "":
    #ra = 0
    #dec = 0



In [11]:
# Get background and zero points from headers:
fits_exposure = {}
fits_zeropoint = {}
fits_zeropointexp = {}
fits_background = {}
#fits_saturation = {}
skyperarcsec = {}

for filter in filters:
    filename = fits_files[filter]
    with fits.open(filename) as hdul:
        header = hdul[0].header 
        data = hdul[1].data
        date_obs = header['DATE-OBS']
        fits_background[filter] = header['BKGLEVEL']
        header = hdul[1].header 
        fits_exposure[filter] = header['XPOSURE']
        fits_zeropoint[filter] = header['ABMAGZP']
        fits_zeropointexp[filter] = header['ABZPEXP']

        # calculate background levels in mag/arcsec^2
        if filter in ['F090W','F150W']:
            skyperarcsec[filter] =  (-2.5 * np.log10(fits_background[filter] / pixscale_sw**2) + fits_zeropoint[filter] )
        elif filter in ['F277W','F356W']:
            skyperarcsec[filter] =  (-2.5 * np.log10(fits_background[filter] / pixscale_lw**2) + fits_zeropoint[filter] )
     

In [12]:
# Write a json file with all the calibration information.

outdict = {}
galaxycenter = []
extinction = []
background = []
morphology = []
velocity = []
exposure = []
zeropoint = []
zeropointexp = []
estimated_seeing = []
input_pixscale = []
estimated_fwhm = []
estimated_saturation = []

outdict["Code version"] = version
#outdict["Date and time run"] = now
outdict["Observation date"] = date_obs
#outdict["Observation date MJD"] = mjd
outdict["RA (2000)"] = ra
outdict["dec (2000)"] = dec
outdict["Exposure time (s)"] = exposure
outdict["AB mag zero points (1/sec)"] = zeropoint
outdict["AB mag zero points (1/exposure)"] = zeropointexp
outdict["Extinction"] = extinction
outdict["Background from the JWST pipeline"] = background
outdict["Morphology flags"] = morphology
outdict["Velocity"] = velocity
outdict["Name"] = name
outdict["Original file base name"] = basename
outdict["Galaxy center location"] = galaxycenter
if pgc!="None":
    outdict["PGC"] = int(pgc)
else:
    outdict["PGC"] = "None"
outdict["Estimated image quality FWHM (arcsec)"] = estimated_seeing
outdict["Pixel scale"] = input_pixscale
outdict["Estimated image quality FWHM (pixels)"] = estimated_fwhm
outdict["Estimated high pixel value limit"] = estimated_saturation
#name.append({"Alternate name": None})
extinction.append({"Extinction Landolt B": float(Bext)})
extinction.append({"Extinction Landolt V": float(Vext)})
extinction.append({"Extinction F090W": float(ext090)})
extinction.append({"Extinction F150W": float(ext150)})
extinction.append({"Extinction F277W": float(ext277)})
extinction.append({"Extinction F356W": float(ext356)})
extinction.append({"Extinction PanSTARRS g": float(psgext)})
extinction.append({"Extinction PanSTARRS z": float(pszext)})
extinction.append({"Extinction DECam g": float(dcgext)})
extinction.append({"Extinction DECam i": float(dciext)})
extinction.append({"Extinction DECam z": float(dczext)})

for filter in filters:
    if filter == 'F090W':
        background.append({"Background F090W (DN)": fits_background[filter]})
        background.append({"Background F090W uncertainty (DN)": None})
        background.append({"Background F090W (mag/arcsec)": skyperarcsec[filter]})
        exposure.append({"Exposure time (F090W)": fits_exposure[filter]})
        zeropoint.append({"F090W AB mag zero point (1 DN/sec)": fits_zeropoint[filter]})
        zeropointexp.append({"F090W AB mag zero point (1 DN/total exposure)": fits_zeropointexp[filter]})
        estimated_seeing.append({"F090W estimated FWHM (arcsec)": seeing[filter]})
        estimated_fwhm.append({"F090W estimated FWHM (pix)": fwhm[filter]})
        input_pixscale.append({"F090W pixel scale (arcsec)": pixscale[filter]})
        estimated_saturation.append({"F090W estimated saturation": maxvalue[filter]})

    elif filter == 'F150W':
        background.append({"Background F150W (DN)": fits_background[filter]})
        background.append({"Background F150W uncertainty (DN)": None})
        background.append({"Background F150W (mag/arcsec)": skyperarcsec[filter]})
        exposure.append({"Exposure time (F150W)": fits_exposure[filter]})
        zeropoint.append({"F150W AB mag zero point (1 DN/sec)": fits_zeropoint[filter]})
        zeropointexp.append({"F150W AB mag zero point (1 DN/total exposure)": fits_zeropointexp[filter]})
        estimated_seeing.append({"F150W estimated FWHM (arcsec)": seeing[filter]})
        estimated_fwhm.append({"F150W estimated FWHM (pix)": fwhm[filter]})
        input_pixscale.append({"F150W pixel scale (arcsec)": pixscale[filter]})
        estimated_saturation.append({"F150W estimated saturation": maxvalue[filter]})

    elif filter == 'F277W':
        background.append({"Background F277W (DN)": fits_background[filter]})
        background.append({"Background F277W uncertainty (DN)": None})
        background.append({"Background F277W (mag/arcsec)": skyperarcsec[filter]})
        exposure.append({"Exposure time (F277W)": fits_exposure[filter]})
        zeropoint.append({"F277W AB mag zero point (1 DN/sec)": fits_zeropoint[filter]})
        zeropointexp.append({"F277W AB mag zero point (1 DN/total exposure)": fits_zeropointexp[filter]})
        estimated_seeing.append({"F277W estimated FWHM (arcsec)": seeing[filter]})
        estimated_fwhm.append({"F277W estimated FWHM (pix)": fwhm[filter]})
        input_pixscale.append({"F277W pixel scale (arcsec)": pixscale[filter]})
        estimated_saturation.append({"F277W estimated saturation": maxvalue[filter]})

    elif filter == 'F356W':
        background.append({"Background F356W (DN)": fits_background[filter]})
        background.append({"Background F356W uncertainty (DN)": None})
        background.append({"Background F356W (mag/arcsec)": skyperarcsec[filter]})
        exposure.append({"Exposure time (F356W)": fits_exposure[filter]})
        zeropoint.append({"F356W AB mag zero point (1 DN/sec)": fits_zeropoint[filter]})
        zeropointexp.append({"F356W AB mag zero point (1 DN/total exposure)": fits_zeropointexp[filter]})
        estimated_seeing.append({"F356W estimated FWHM (arcsec)": seeing[filter]})
        estimated_fwhm.append({"F356W estimated FWHM (pix)": fwhm[filter]})
        input_pixscale.append({"F356W pixel scale (arcsec)": pixscale[filter]})
        estimated_saturation.append({"F356W estimated saturation": maxvalue[filter]})

velocity.append({"Heliocentric velocity": v})
velocity.append({"CMB frame velocity": v3k})
velocity.append({"LG frame velocity": vlg})
outdict["Pixel scale SW"] = pixscale_sw
outdict["Pixel scale LW"] = pixscale_lw
outdict["Renuc factor for likenew"] = renuc
galaxycenter.append({"Original center x SW": int(xcenter_sw)})
galaxycenter.append({"Original center y SW": int(ycenter_sw)})
galaxycenter.append({"Original center x LW": int(xcenter_lw)})
galaxycenter.append({"Original enter y LW": int(ycenter_lw)})
galaxycenter.append({"Cutout center x SW": int(new_xcen_sw)})
galaxycenter.append({"Cutout center y SW": int(new_ycen_sw)})
galaxycenter.append({"Cutout center x LW": int(new_xcen_lw)})
galaxycenter.append({"Cutout center y LW": int(new_ycen_lw)})
outdict["Gain e/DN"] = gain
morphology.append({"T-type": ttype})
morphology.append({"Galaxy type": gtype})
morphology.append({"Ring": ring})
morphology.append({"Bar": bar})
morphology.append({"Companion(s)": companion})
morphology.append({"Nuclear dust": ""})
morphology.append({"Dust in the SBF region": ""})
morphology.append({"Shells": ""})
morphology.append({"Galaxy model subtraction residuals": ""})
outdict["Approximate distance (Mpc)"] = int(distance)
   
json_name = datafile
with open(json_name, 'w') as file:
    json_string = json.dumps(outdict, default=lambda o: o.__dict__, sort_keys=True, indent=2)
    file.write(json_string)

print(f"Data for {name} are stored in {json_name}")

    #return json_name


Data for n1380 are stored in /Users/joejensen/data/jwst-3055/SBF/n1380/n1380_data.json


In [13]:
!cat {json_name}

{
  "AB mag zero points (1/exposure)": [
    {
      "F090W AB mag zero point (1 DN/total exposure)": 37.85586698455117
    },
    {
      "F150W AB mag zero point (1 DN/total exposure)": 36.30733375170959
    },
    {
      "F277W AB mag zero point (1 DN/total exposure)": 36.33508429102382
    },
    {
      "F356W AB mag zero point (1 DN/total exposure)": 34.78651175932431
    }
  ],
  "AB mag zero points (1/sec)": [
    {
      "F090W AB mag zero point (1 DN/sec)": 27.999457502628196
    },
    {
      "F150W AB mag zero point (1 DN/sec)": 27.999508891414784
    },
    {
      "F277W AB mag zero point (1 DN/sec)": 26.478674809100845
    },
    {
      "F356W AB mag zero point (1 DN/sec)": 26.478686899029505
    }
  ],
  "Approximate distance (Mpc)": 24,
  "Background from the JWST pipeline": [
    {
      "Background F090W (DN)": 0.18027129769325256
    },
    {
      "Background F090W uncertainty (DN)": null
    },
    {
      "Background F090W (mag/arcsec)": 22.316449509204972
   

### These are the files used by the original SBF software. 

In [14]:
# make calibrate.dat (needs sky value, zero point, computes zero point); centers.dat (center, zero point)
# This is for backwards compatibility.

file_content = f"""GAIN_*  =                 2.00  /  e/ADU for raw images
PATOP_* =                  0.0  /  Position angle of top of image
PALEFT_*=                  0.0  /  Position angle of left of image
SECPIX_*=                0.031  /  Image scale: arcseconds per pixel
GALAXY_*= '{name}'               /  Galaxy name
GALEXT_*=                {Bext:.3f}  /  Galactic extinction (B band)
REDSHIFT=                 {v:.0f}  /  Heliocentric redshift (cz, km/s)
SECPIX_C=                {pixscale['F090W']:.3f}  /  Image scale: arcseconds per pixel
M1_C    =               {fits_zeropoint['F090W']:.3f}  /  m1: m for 1 e/sec at top of atm, color=0
ATMEXT_C=                0.000  /  Atmospheric extinction: mag/airmass
CTERM_C =                0.000  /  Color term: m=-2.5logf+m_1-A*secz+C*color
COLOR_C = ''                    /  Color used for color term
ETIME_C =               {fits_exposure['F090W']:.1f}  /  Exposure time (sec)
SECZ_C  =                0.000  /  Airmass
E/ADU_C =                  {gain}  /  Electrons per ADU in averaged image
IMAGES_C= ''                    /  Original images comprising summed image
SKY_C   =                {fits_background['F090W']:.3f}  /  Sky brightness (e/pixel)
SEEING_C=                {seeing["F090W"]:0.3f} /  PSF FWHM (arcsec)
M1STAR_C=               {fits_zeropointexp['F090W']:.3f}  /  m1star: m for 1e- net (no MW ext, C=gxy)
SKYMAG_C=                 {skyperarcsec['F090W']:.1f}  /  Sky brightness (mag/arcsec)
SECPIX_D=                {pixscale['F150W']:.3f}  /  Image scale: arcseconds per pixel
M1_D    =               {fits_zeropoint['F150W']:.3f}  /  m1: m for 1 e/sec at top of atm, color=0
ATMEXT_D=                0.000  /  Atmospheric extinction: mag/airmass
CTERM_D =                0.000  /  Color term: m=-2.5logf+m_1-A*secz+C*color
COLOR_D = ''                    /  Color used for color term
ETIME_D =               {fits_exposure['F150W']:.1f}  /  Exposure time (sec)
SECZ_D  =                0.000  /  Airmass
E/ADU_D =                  {gain}  /  Electrons per ADU in averaged image
IMAGES_D= ''                    /  Original images comprising summed image
SKY_D   =                {fits_background['F150W']:.3f}  /  Sky brightness (e/pixel)
SEEING_D=                {seeing["F150W"]:0.3f}  /  PSF FWHM (arcsec)
M1STAR_D=               {fits_zeropointexp['F150W']:.3f}  /  m1star: m for 1e- net (no MW ext, C=gxy)
SKYMAG_D=                {skyperarcsec['F150W']:.2f}  /  Sky brightness (mag/arcsec)
SECPIX_E=                {pixscale['F277W']:.3f}  /  Image scale: arcseconds per pixel
M1_E    =               {fits_zeropoint['F277W']:.3f}  /  m1: m for 1 e/sec at top of atm, color=0
ATMEXT_E=                0.000  /  Atmospheric extinction: mag/airmass
CTERM_E =                0.000  /  Color term: m=-2.5logf+m_1-A*secz+C*color
COLOR_E = ''                    /  Color used for color term
ETIME_E =               {fits_exposure['F277W']:.1f}  /  Exposure time (sec)
SECZ_E  =                0.000  /  Airmass
E/ADU_E =                  {gain}  /  Electrons per ADU in averaged image
IMAGES_E= ''                    /  Original images comprising summed image
SKY_E   =                {fits_background['F277W']:.3f}  /  Sky brightness (e/pixel)
SEEING_E=                {seeing["F277W"]:0.3f}  /  PSF FWHM (arcsec)
M1STAR_E=               {fits_zeropointexp['F277W']:.3f}  /  m1star: m for 1e- net (no MW ext, C=gxy)
SKYMAG_E=                {skyperarcsec['F277W']:.2f}  /  Sky brightness (mag/arcsec)
SECPIX_F=                {pixscale['F356W']:.3f}  /  Image scale: arcseconds per pixel
M1_F    =               {fits_zeropoint['F356W']:.3f}  /  m1: m for 1 e/sec at top of atm, color=0
ATMEXT_F=                0.000  /  Atmospheric extinction: mag/airmass
CTERM_F =                0.000  /  Color term: m=-2.5logf+m_1-A*secz+C*color
COLOR_F = ''                    /  Color used for color term
ETIME_F =               {fits_exposure['F356W']:.1f}  /  Exposure time (sec)
SECZ_F  =                0.000  /  Airmass
E/ADU_F =                  {gain}  /  Electrons per ADU in averaged image
IMAGES_F= ''                    /  Original images comprising summed image
SKY_F   =                {fits_background['F356W']:.3f}  /  Sky brightness (e/pixel)
SEEING_F=                {seeing["F356W"]:0.3f}  /  PSF FWHM (arcsec)
M1STAR_F=               {fits_zeropointexp['F356W']:.3f}  /  m1star: m for 1e- net (no MW ext, C=gxy)
SKYMAG_F=                {skyperarcsec['F356W']:.2f}  /  Sky brightness (mag/arcsec)
END
"""

if not os.path.exists(calibrate) or overwrite:
    with open(calibrate, "w") as file:
        file.write(file_content)
    print(f"New {calibrate} for {name} generated.")
else:
    print(f"Original {calibrate} for {name} retained.")

New /Users/joejensen/data/jwst-3055/SBF/n1380/calibrate.dat for n1380 generated.


In [15]:
!cat {calibrate}

GAIN_*  =                 2.00  /  e/ADU for raw images
PATOP_* =                  0.0  /  Position angle of top of image
PALEFT_*=                  0.0  /  Position angle of left of image
SECPIX_*=                0.031  /  Image scale: arcseconds per pixel
GALAXY_*= 'n1380'               /  Galaxy name
GALEXT_*=                0.073  /  Galactic extinction (B band)
REDSHIFT=                 1874  /  Heliocentric redshift (cz, km/s)
SECPIX_C=                0.031  /  Image scale: arcseconds per pixel
M1_C    =               27.999  /  m1: m for 1 e/sec at top of atm, color=0
ATMEXT_C=                0.000  /  Atmospheric extinction: mag/airmass
CTERM_C =                0.000  /  Color term: m=-2.5logf+m_1-A*secz+C*color
COLOR_C = ''                    /  Color used for color term
ETIME_C =               8761.2  /  Exposure time (sec)
SECZ_C  =                0.000  /  Airmass
E/ADU_C =                  1.0  /  Electrons per ADU in averaged image
IMAGES_C= ''                    /  Origi

In [18]:
# make the centers.dat file
# This is for backwards compatability.
file_content = ""
for filter in filters:
    if filter in ['F090W','F150W']:
        xcenter = new_xcen_sw; ycenter = new_ycen_sw
    elif filter in ['F277W','F356W']:
        xcenter = new_xcen_lw; ycenter = new_ycen_lw
    file_content += f"{name}{filterlabel[filter]}  {xcenter:.0f}  {ycenter:.0f}  {fits_zeropointexp[filter]:.3f}\n"
    
if not os.path.exists(centers) or overwrite:
    with open(centers, "w") as file:
        file.write(file_content)
    print(f"New {centers} for {name} generated.")
else:
    print(f"Original {centers} for {name} retained.")

New /Users/joejensen/data/jwst-3055/SBF/n1380/centers.dat for n1380 generated.


In [19]:
!cat {centers}

n1380c  1024  1024  37.856
n1380d  1024  1024  36.307
n1380e  512  512  36.335
n1380f  512  512  34.787


In [20]:
# make flucin file, dpar file for Dophot, and inpar file for SExtractor
# This is for backwards compatability.

for f in filterlabel:
    if f in ['F090W','F150W']:
        xcenter = new_xcen_sw; ycenter = new_ycen_sw
        proversion = "_sw"
    elif f in ['F277W','F356W']:
        xcenter = new_xcen_lw; ycenter = new_ycen_lw
        proversion = "_lw"
    saturation = int(0.95 * maxvalue[f] * fits_exposure[f])
    sky = fits_background[f] * fits_exposure[f]
    
    if not os.path.exists(flucin[f]) or overwrite:
        args = f"{name} {int(xcenter)} {int(ycenter)} {sky} {int(saturation)} {int(distance)} {renuc} {fits_zeropointexp[f]} {fits_exposure[f]} {filterlabel[f]} {SBF_root} {config_path} {proversion}"
        ! {config_path}jwstsbfsetup.sh {args}
        print(f"New SBF file {flucin[f]} for {name} generated.")
    else:
        print(f"Original {flucin[f]} for {name} retained.")

    if not os.path.exists(dpar[f]) or overwrite:
        args = f"{name}  {fits_background[f]}  {saturation} {fits_zeropointexp[f]} {filterlabel[f]} {fwhm[f]} {pixscale[f]} {seeing[f]} {SBF_root} {config_path} {proversion}"
        ! {config_path}jwstsedosetup.sh {args}
        print(f"New SExtractor and Dophot configuration files written for {name}{filterlabel[f]}.")
    else:
        print(f"Original {dpar} and {inpar} retained.")    

New SBF file /Users/joejensen/data/jwst-3055/SBF/n1380/n1380c.flucin for n1380 generated.
New SExtractor and Dophot configuration files written for n1380c.
New SBF file /Users/joejensen/data/jwst-3055/SBF/n1380/n1380d.flucin for n1380 generated.
New SExtractor and Dophot configuration files written for n1380d.
New SBF file /Users/joejensen/data/jwst-3055/SBF/n1380/n1380e.flucin for n1380 generated.
New SExtractor and Dophot configuration files written for n1380e.
New SBF file /Users/joejensen/data/jwst-3055/SBF/n1380/n1380f.flucin for n1380 generated.
New SExtractor and Dophot configuration files written for n1380f.


In [ ]:
# This is not needed for JWST; sky was already subtracted.
#if not os.path.exists(SBF_root+'/'+name+'/skydata'):
#    ! mkdir {SBF_root+'/'+name+'/skydata'}

In [ ]:
now = datetime.now()
print(now)